# Tasks - Genomics Part 1: From raw reads to alignments

<div class="alert alert-info" style="background-color:rgba(212, 240, 255, 0.5); border-color:rgba(212, 240, 255, 1)">
Use this notebook as a template to solve the following tasks and to run the whole alignment pipeline. 

- Use Markdown cells for headings and explanations
- Use code cells to run bash commands with the `!` prefix or the `%%bash` magic command. **Don't forget to actually run the commands by pressing **Shift+Enter** or **Ctrl+Enter**!**
- **Some code cells can be run as they are without any modification, but when a task asks you to write the relevant code, make sure to include it in the prepared code cell below.**
- Provide explanations and comments in the code cells by starting them with `#`
- For screenshot tasks, create a screenshot, upload it to the `practical3` folder and insert the image in the Markdown cell

</div>

## Setup and data exploration

### Working directory structure

In [ ]:
%%bash

# Create main project directory
mkdir -p genomics_practical3

# Create subdirectories for different stages
cd genomics_practical3
mkdir -p data/fastq
mkdir -p data/reference
mkdir -p results/fastqc
mkdir -p results/bam
mkdir -p results/bam_processed
mkdir -p results/logs

# Check the structure
tree -L 2

### Copying the FASTQ files

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 1

Copy all files ending with .fastq.gz from the shared data folder to the local data folder (`genomics_practical3/data/fastq/`).

</div>

In [ ]:
%%bash

# Set the shared data path
SHARED_DATA="/shared/OMICS/practical_03/data/"

# Copy all files ending with .fastq.gz from the shared data folder to the local data folder
# write your command here


# Verify files were copied
ls -lh genomics_practical3/data/fastq/

### Exploring the FASTQ files

In [ ]:
%%bash

# Navigate to FASTQ directory
cd genomics_practical3/data/fastq/

# Check file sizes
ls -lh

#### Understanding FASTQ Format

In [ ]:
%%bash

cd genomics_practical3/data/fastq/

# View first read from Patient 1 adenoma sample
zcat P1_AD_TP53_R1.fastq.gz | head -n 4

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 2

How much more accurate is a base call of quality E compared to A?

</div>

**Your answer:**

#### How many reads do we have?

In [ ]:
%%bash

cd genomics_practical3/data/fastq/

# Count total number of lines in one FASTQ file (each read entry has 4 lines)
zcat P1_AD_TP53_R1.fastq.gz | wc -l

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 3

1. Calculate the number of reads in Patient 1’s NAT sample (both R1 and R2).
2. Should R1 and R2 files from the same sample have the same number of reads? Why?
3. How does this number compare to a typical WES dataset? (Hint: WES = ~50-100 million reads)

</div>

In [ ]:
%%bash

# Count reads in R1
# write your command here

# Count reads in R2
# write your command here

**Your answers:**

- For 3/1 read numbers:
   - P1_AD_TP53_R1.fastq.gz: _________________ reads
   - P1_AD_TP53_R2.fastq.gz: _________________ reads
- For 3/2:
- For 3/3:

## Quality control with FastQC

### Running FastQC on one sample

In [ ]:
%%bash

# Run FastQC on Patient 1 adenoma sample
cd genomics_practical3

fastqc \
  data/fastq/P1_AD_TP53_R1.fastq.gz \
  data/fastq/P1_AD_TP53_R2.fastq.gz \
  --outdir results/fastqc/ \
  --threads 2

### Interpreting FastQC results

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 4

Open the FastQC report for P1_AD_TP53_R1.fastq.gz. Answer the following:

1. What is the read length? _________________ bp
2. What is the total number of sequences? _________________
3. What is the %GC content? _________________ % (TP53 has ~42% GC)
4. Are there any failed quality modules (red ❌)? If so, which ones?
5. Based on Per base sequence quality, do the reads need trimming?
6. Is there adapter contamination that needs to be removed?

</div>

**Your answers:**

- For 4/1:
- For 4/2:
- For 4/3:
- For 4/4:
- For 4/5:
- For 4/6:

### Running FastQC on all samples

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 5

Run FastQC on all samples.

</div>

In [ ]:
%%bash
cd genomics_practical3

# Run FastQC on all samples
# write your command here


# This should create 32 output files (16 HTML + 16 ZIP), check that they were created successfully
ls -lh results/fastqc/

### Aggregate report with MultiQC

In [ ]:
%%bash
cd genomics_practical3

# Run MultiQC (combines all FastQC reports)
multiqc results/fastqc/ --outdir results/fastqc/

## Reference genome preparation

### Downloading the reference genome

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 6

Download the reference genome for chromosome 17 from Ensembl.org to your reference directory.

</div>

In [ ]:
%%bash

# Navigate to reference directory
cd genomics_practical3/data/reference/

# Find the URL for the chromosome 17 FASTA from Ensembl.org
# and download it to your reference directory with the name `chr17.fa.gz`
# write your command here

# decompress the file
gunzip chr17.fa.gz

### Examining the reference genome

In [ ]:
%%bash

cd genomics_practical3/data/reference/

# View the first few lines
head -n 10 chr17.fa

In [ ]:
%%bash
cd genomics_practical3/data/reference/

# Count lines in the file
wc -l chr17.fa

# Get file size
ls -lh chr17.fa

### Indexing the reference genome

In [ ]:
%%bash

# Index the reference genome with BWA
cd genomics_practical3/data/reference/

bwa index chr17.fa

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 7

1. Check that all 5 index files were created successfully (.amb, .ann, .bwt, .pac, .sa) in your reference directory.
2. What is the size of the reference FASTA file compared to the combined size of all index files? Why do you think the index files require significant space?

</div>

In [ ]:
%%bash

# write your code here


**Your answer:**

- For 7/2:

## Read alignment with BWA

### Aligning one sample

In [ ]:
%%bash
# Navigate to project directory
cd genomics_practical3

# Run BWA-MEM on Patient 1 adenoma sample
bwa mem \
  -t 4 \
  -R '@RG\tID:P1_AD\tSM:P1_Adenoma\tPL:ILLUMINA\tLB:TP53_targeted' \
  data/reference/chr17.fa \
  data/fastq/P1_AD_TP53_R1.fastq.gz \
  data/fastq/P1_AD_TP53_R2.fastq.gz \
  > results/bam/P1_AD_TP53.sam

### Understanding the alignment output (SAM format)

**Header section:**

In [ ]:
%%bash
grep "^@"  genomics_practical3/results/bam/P1_AD_TP53.sam

**Alignment section:**

In [ ]:
%%bash
grep -v "^@"  genomics_practical3/results/bam/P1_AD_TP53.sam | head -n 3

**Number of aligned reads:**

In [ ]:
%%bash

cd genomics_practical3/results/bam/

# Count total alignments
grep -v "^@" P1_AD_TP53.sam | wc -l

## SAM to BAM Conversion

### Converting SAM to BAM

In [ ]:
%%bash

cd genomics_practical3/

# Convert SAM to BAM
samtools view \
  -b \
  -h \
  -o results/bam/P1_AD_TP53.bam \
  results/bam/P1_AD_TP53.sam

# Parameters:
# -b = output BAM format (binary)
# -h = include header in output
# -o = output file path

### Checking alignment statistics

In [ ]:
%%bash

cd genomics_practical3/results/bam/

# Get comprehensive alignment statistics
samtools flagstat P1_AD_TP53.bam

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 8

Run `samtools flagstat` on the `P1_AD_TP53.bam` file and record:

- Total number of reads: _________________
- Percentage mapped: _________________
- Percentage properly paired: _________________
- Number of singletons: _________________

1. What does properly paired mean?
2. What is a good mapping rate?
3. What are singletons and why do they occur?
</div>

**Your answers:**

- Total number of reads: _________________
- Percentage mapped: _________________
- Percentage properly paired: _________________
- Number of singletons: _________________

- For 8/1:
- For 8/2:
- For 8/3:

### Remove the SAM file

In [ ]:
%%bash
cd genomics_practical3/results/bam/

# Check file sizes
ls -lh P1_AD_TP53.bam
ls -lh P1_AD_TP53.sam

# Remove SAM file to save disk space
rm P1_AD_TP53.sam

## Post-processing BAM files

### Sorting BAM files

In [ ]:
%%bash

cd genomics_practical3/results/

# Sort BAM file by genomic coordinate
samtools sort \
  -@ 4 \
  -o bam_processed/P1_AD_TP53_sorted.bam \
  bam/P1_AD_TP53.bam

# Parameters:
# -@ 4 = use 4 threads
# -o = output file

In [ ]:
%%bash

cd genomics_practical3/results/

# Compare file sizes (sorting may compress slightly better)
ls -lh bam/P1_AD_TP53.bam
ls -lh bam_processed/P1_AD_TP53_sorted.bam

### Marking duplicates

In [ ]:
%%bash

cd genomics_practical3/results

# Mark duplicates with Picard
picard MarkDuplicates \
  I=bam_processed/P1_AD_TP53_sorted.bam \
  O=bam_processed/P1_AD_TP53_sorted_dedup.bam \
  M=logs/P1_AD_TP53_dedup_metrics.txt \
  REMOVE_DUPLICATES=false

# Parameters:
# I = input BAM
# O = output BAM (with duplicates flagged)
# M = metrics file (statistics)
# REMOVE_DUPLICATES=false - mark duplicates but keep them in the file (useful for QC)

In [ ]:
%%bash

cd genomics_practical3/results

# View duplication metrics
cat logs/P1_AD_TP53_dedup_metrics.txt

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 9

Check the `logs/P1_AD_TP53_dedup_metrics.txt` file.

1. What percentage of reads are marked as duplicates?
2. Is this level acceptable for targeted sequencing?
3. Would you expect more or fewer duplicates in whole genome sequencing? Why?
</div>

**Your answers:**

- For 9/1:
- For 9/2:
- For 9/3:

### Indexing BAM files

In [ ]:
%%bash

cd genomics_practical3/results/bam_processed/

# Index the final processed BAM file
samtools index P1_AD_TP53_sorted_dedup.bam

# This creates: P1_AD_TP53_sorted_dedup.bam.bai

# Verify both files exist
ls -lh P1_AD_TP53_sorted_dedup.bam*

## Processing all samples

In [ ]:
%%bash

cd genomics_practical3

# Create the processing script
cat > process_all_samples.sh << 'EOF'
#!/bin/bash

# Script to process all TP53 targeted sequencing samples
# Runs: BWA alignment → SAM to BAM → Sort → Mark Duplicates → Index

set -e  # Exit on any error

# Configuration
REF="data/reference/chr17.fa"
FASTQ_DIR="data/fastq"
THREADS=4

# Sample definitions: Patient_SampleType
# Samples: P1_AD, P1_NAT, P2_AD, P2_NAT, P3_CRC, P3_NAT, P4_CRC, P4_NAT
SAMPLES=(
  "P1_AD:P1_Adenoma"
  "P1_NAT:P1_Normal"
  "P2_AD:P2_Adenoma"
  "P2_NAT:P2_Normal"
  "P3_CRC:P3_Cancer"
  "P3_NAT:P3_Normal"
  "P4_CRC:P4_Cancer"
  "P4_NAT:P4_Normal"
)

echo "========================================"
echo "Starting processing of ${#SAMPLES[@]} samples"
echo "========================================"
echo ""

# Process each sample
for SAMPLE_INFO in "${SAMPLES[@]}"; do
  # Parse sample ID and name
  IFS=':' read -r SAMPLE SAMPLE_NAME <<< "$SAMPLE_INFO"
  
  echo "========================================="
  echo "Processing: $SAMPLE ($SAMPLE_NAME)"
  echo "========================================="
  
  # Step 1: Alignment with BWA-MEM
  echo "[1/5] Aligning reads with BWA-MEM..."
  bwa mem \
    -t $THREADS \
    -R "@RG\tID:${SAMPLE}\tSM:${SAMPLE_NAME}\tPL:ILLUMINA\tLB:TP53_targeted" \
    $REF \
    ${FASTQ_DIR}/${SAMPLE}_TP53_R1.fastq.gz \
    ${FASTQ_DIR}/${SAMPLE}_TP53_R2.fastq.gz \
    > results/bam/${SAMPLE}_TP53.sam
  
  # Step 2: SAM to BAM conversion
  echo "[2/5] Converting SAM to BAM..."
  samtools view -b -h -o results/bam/${SAMPLE}_TP53.bam results/bam/${SAMPLE}_TP53.sam
  rm results/bam/${SAMPLE}_TP53.sam  # Remove SAM to save space
  
  # Step 3: Sort BAM by coordinate
  echo "[3/5] Sorting BAM..."
  samtools sort \
    -@ $THREADS \
    -o results/bam_processed/${SAMPLE}_TP53_sorted.bam \
    results/bam/${SAMPLE}_TP53.bam
  
  # Step 4: Mark PCR duplicates
  echo "[4/5] Marking duplicates..."
  picard MarkDuplicates \
    I=results/bam_processed/${SAMPLE}_TP53_sorted.bam \
    O=results/bam_processed/${SAMPLE}_TP53_sorted_dedup.bam \
    M=results/logs/${SAMPLE}_TP53_dedup_metrics.txt \
    REMOVE_DUPLICATES=false \
    2>&1 | grep -E "(Marking|MarkDuplicates|Duplicate)" | head -5
  
  # Step 5: Index final BAM
  echo "[5/5] Indexing BAM..."
  samtools index results/bam_processed/${SAMPLE}_TP53_sorted_dedup.bam
  
  # Generate flagstats
  samtools flagstat results/bam_processed/${SAMPLE}_TP53_sorted_dedup.bam \
    > results/logs/${SAMPLE}_TP53_flagstat.txt
  
  echo "✓ Sample $SAMPLE complete!"
  echo ""
done

echo "========================================="
echo "All samples processed successfully!"
echo "========================================="
echo ""
echo "Summary:"
ls -lh results/bam_processed/*_TP53_sorted_dedup.bam
echo ""
echo "Ready for variant calling!"
EOF

# Make script executable
chmod +x process_all_samples.sh

In [ ]:
%%bash

cd genomics_practical3

# Run the script
./process_all_samples.sh 2>&1 | tee processing_log.txt

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 10

If you wanted to run fastqc on all samples, how would you change the above script? Write down the commands you would add and where you would add them.
</div>

**Your answer:**

## Quality assessment

### Coverage statistics

In [ ]:
%%bash

cd genomics_practical3/results/

# TP53 gene coordinates: chr17:7,661,779-7,687,550
# Calculate coverage for Patient 1 adenoma
samtools depth \
  -r 17:7661779-7687550 \
  bam_processed/P1_AD_TP53_sorted_dedup.bam \
  > logs/P1_AD_TP53_coverage.txt

# Calculate mean coverage
awk '{sum+=$3; count++} END {print "Mean coverage:", sum/count}' \
  logs/P1_AD_TP53_coverage.txt

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 11

Calculate mean coverage for Patient 3’s CRC and normal tissue samples.

- P3 CRC mean coverage:
- P3 NAT mean coverage:

1. Are these coverage values sufficient for variant calling?
2. Do you see differences between adenoma and NAT samples?
</div>

In [ ]:
%%bash

# write your code here for calculating the mean coverage of the P3 CRC sample


# write your code here for calculating the mean coverage of the P3 NAT sample

**Your answers:**
- For 11/1:
- For 11/2:

### Summary statistics for all samples

In [ ]:
%%bash

cd genomics_practical3

# Create summary table
echo "Sample,Total_Reads,Mapped,Mapped_%,Properly_Paired,Properly_Paired_%,Duplicates_%" \
  > results/summary_stats.csv

for SAMPLE in P1_AD P1_NAT P2_AD P2_NAT P3_CRC P3_NAT P4_CRC P4_NAT; do
  # Parse flagstat file
  FLAGSTAT="results/logs/${SAMPLE}_TP53_flagstat.txt"
  
  TOTAL=$(grep "in total" $FLAGSTAT | awk '{print $1}')
  MAPPED=$(grep "mapped (" $FLAGSTAT | head -1 | awk '{print $1}')
  MAPPED_PCT=$(grep "mapped (" $FLAGSTAT | head -1 | awk -F'[(%]' '{print $2}')
  PAIRED=$(grep "properly paired" $FLAGSTAT | awk '{print $1}')
  PAIRED_PCT=$(grep "properly paired" $FLAGSTAT | awk -F'[(%]' '{print $2}')
  
  # Get duplicates from metrics file
  DUP=$(grep -A 2 "## METRICS" results/logs/${SAMPLE}_TP53_dedup_metrics.txt | \
    tail -1 | awk '{printf "%.2f", ($7/$3)*100}')
  
  echo "$SAMPLE,$TOTAL,$MAPPED,$MAPPED_PCT,$PAIRED,$PAIRED_PCT,$DUP" \
    >> results/summary_stats.csv
done

# View the summary
column -t -s',' results/summary_stats.csv

## Verification in IGV

<div class="alert alert-warning" style="background-color:rgba(255, 227, 194, 0.5); border-color:rgba(255, 227, 194, 1); color:rgba(0, 0, 0, 0.8)">

## Task 12

Load the reference genome FASTA file (located in `genomics_practical3/data/reference/chr17.fa`) and the transcript annotation .bed file for TP53 (located in `/shared/OMICS/practical_03/TP53_transcripts.bed`) into IGV.

Load the BAM files of the two samples (healthy and diseased) of Patient 4.

Navigate to the TP53 gene region and examine the alignments. Answer these questions:

1. Do you see higher coverage over exons compared to introns?
2. Are coverage levels consistent with your calculated mean coverage?
3. Are read pairs properly oriented (arrows pointing inward: → ← )?
4. Do you see any positions where many reads have the same mismatch in the healthy sample AND the diseased sample as well? (These could be real germline variants!)
5. Do you see any positions with mismatches in the diseased sample but NOT in the healthy one? (These could be somatic mutations!)
6. Take a screenshot showing:
   - Both healthy and diseased samples loaded
   - Coverage track visible
   - Zoomed to show individual reads and potential variants
   - Transcript annotation visible

**Insert your screenshot into the Jupyter Notebook.**
</div>

**Your answers:**

- For 12/1:
- For 12/2:
- For 12/3:
- For 12/4:
- For 12/5:
- For 12/6:

<!-- Upload your screenshot to the practical3 folder and use markdown to insert it:
![IGV Screenshot](screenshot_name.png)
-->

---

## Congratulations!

You've completed all the tasks for Genomics Part 1 practical.

**Before submitting:**
1. Make sure all code cells have been run (Shift+Enter)
2. Check that all your answers are filled in
3. Save this notebook (Ctrl+S or Cmd+S)
4. Make sure this notebook is in your `practical3` folder
5. Name it: `firstname_lastname_practical3.ipynb`

**To download your notebook:**
- File → Download as → Notebook (.ipynb)